# Loading in Multi-Label BERT Clasifier

* Load in the trained BERT model from 'multi_label_bert.ipynb'

In [1]:
import torch 
import pickle 

from transformers import BertTokenizer, BertForSequenceClassification

/Users/svenwu/Hustle/MachineLearning/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# We do not have to retrain the model.
model_path = "./bert_multilabel_v1"

tokenizer = BertTokenizer.from_pretrained(model_path)

model = BertForSequenceClassification.from_pretrained(model_path)

model.eval()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4851.56it/s]


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [5]:
with open("./bert_multilabel_v1/emotion_names.pkl", "rb") as f:
    emotion_names = pickle.load(f)

In [6]:
# Test on a new sentence.

text = "I absolutely love this game!"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=64
)

with torch.no_grad():
    outputs = model(**inputs)

probs = torch.sigmoid(outputs.logits)          # sigmoid, not softmax — independent per-label probabilities
preds = (probs > 0.3).int().squeeze()          # threshold 0.3, matches your chosen final config

predicted_indices = torch.where(preds == 1)[0].tolist()
predicted_emotions = [emotion_names[i] for i in predicted_indices]

print("Prediction:", predicted_emotions)
print("Probabilities:", {emotion_names[i]: round(probs[0][i].item(), 3) for i in predicted_indices})

Prediction: ['love']
Probabilities: {'love': 0.933}


In [7]:
def predict_emotion(text, model, tokenizer, emotion_names, max_length=64, threshold=0.3):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.sigmoid(outputs.logits).squeeze()      # independent per-label probabilities
    preds = (probs > threshold).int()

    predicted_indices = torch.where(preds == 1)[0].tolist()
    predicted_emotions = [emotion_names[i] for i in predicted_indices]

    return predicted_emotions

In [8]:
predict_emotion("I absolutely love this game!", model, tokenizer, emotion_names)
# e.g. ['admiration', 'love']

['love']

In [9]:
predict_emotion("it's horrid :/	", model, tokenizer, emotion_names)

['fear']

In [10]:
predict_emotion("i love and hate this game.", model, tokenizer, emotion_names)

['love']

In [11]:
predict_emotion("i hate and love this game.", model, tokenizer, emotion_names)

['love']

In [12]:
predict_emotion("I'm so grateful but also a little nervous about tomorrow", model, tokenizer, emotion_names)
# plausibly: ['gratitude', 'nervousness']

['gratitude']